<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/04_NeuroFHIR_QC_Segmentation_and_Volumetry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/04_NeuroFHIR_QC_Segmentation_and_Volumetry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeuroFHIR-QC — Notebook 04
## Pretrained Brain-Tumor Segmentation, Volumetry, and Reference-Mask Evaluation

**Notebook filename:** `04_NeuroFHIR_QC_Segmentation_and_Volumetry.ipynb`  
**Project root:** `/content/drive/MyDrive/neurofhir-qc`  
**Required runtime:** Google Colab with a GPU  
**Imaging source:** the three public de-identified MSD Task01 cases prepared in Notebook 03  
**FHIR context:** synthetic demonstration records only

### What this notebook builds

This notebook executes the first actual model-inference stage of NeuroFHIR-QC. It:

1. enforces the Notebook 03 imaging-evidence gate;
2. reloads the three prepared four-modality MRI cases and their expert reference masks;
3. obtains the pinned MONAI `brats_mri_segmentation` model artifact;
4. supplies the model modalities in its documented order: **T1-contrast, T1, T2, FLAIR**;
5. runs 3D sliding-window inference;
6. saves tumor-core, whole-tumor, enhancing-tumor, and reconstructed multiclass masks;
7. calculates predicted volume in milliliters;
8. measures Dice, sensitivity, precision, HD95, absolute volume error, and relative volume error;
9. creates overlays, structured result manifests, reusable inference code, checksums, and an execution audit.

### Interpretation boundary

This is an **academic research demonstration**, not clinical validation. The public MSD cases may overlap in source lineage with data used to train the published BraTS model, so the three-case results must **not** be described as independent external validation. They are an executable demonstration benchmark showing that the imaging-to-segmentation-to-volumetry segment works and produces measurable artifacts.

Notebook 04 does **not** assign the final NeuroFHIR-QC confidence category, perform perturbation robustness testing, create the current FHIR AI Observation, finalize a result, perform human review, or write AI evidence back to FHIR. Those stages remain for Notebooks 05–08.

### Model sources recorded by this notebook

- MONAI model-zoo bundle: `brats_mri_segmentation`, version `0.5.4`, pinned commit `370f7f9d062745fbac445e7fe6d6616d35df04ec`
- MONAI model documentation: `https://github.com/Project-MONAI/model-zoo/tree/dev/models/brats_mri_segmentation`
- Hosted model repository: `https://huggingface.co/MONAI/brats_mri_segmentation`
- Pinned checkpoint SHA-256: `860ccb3f1c21c99d0410ad8a1ac4ef6b8fab60cec0a503b0ba42675741a750ae`
- Intended use from the MONAI metadata: example/research use, not diagnostic use

Run the cells in order. Do not skip the final audit cell.

In [1]:
# Cell 1 — Mount Drive and enforce the Notebook 03 imaging-evidence gate

from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
import textwrap
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Open this notebook in Google Colab.") from exc

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")
CONFIG_PATH = PROJECT_ROOT / "project_config.json"
NOTEBOOK_MANIFEST_PATH = PROJECT_ROOT / "notebook_manifest.json"
NOTEBOOK_03_AUDIT_PATH = (
    PROJECT_ROOT / "evaluation/results/notebook_03_imaging_preparation_audit.json"
)
IMAGING_MANIFEST_PATH = (
    PROJECT_ROOT / "data/sample_images/notebook_03/imaging_case_manifest.json"
)
IMAGING_INTEGRITY_PATH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_03_imaging_preparation/imaging_integrity_report.json"
)


def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False, allow_nan=False)
        handle.write("\n")
    temporary.replace(path)


def utc_now() -> str:
    return (
        datetime.now(timezone.utc)
        .replace(microsecond=0)
        .isoformat()
        .replace("+00:00", "Z")
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def md5_file(path: Path) -> str:
    try:
        digest = hashlib.md5(usedforsecurity=False)
    except TypeError:
        digest = hashlib.md5()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def notebook_entries(manifest: Any) -> list[dict[str, Any]]:
    if isinstance(manifest, list):
        return manifest
    if isinstance(manifest, dict):
        for key in ("notebooks", "entries", "workflow"):
            value = manifest.get(key)
            if isinstance(value, list):
                return value
    raise ValueError("Unrecognized notebook_manifest.json structure.")


def normalize_number(value: Any) -> str:
    match = re.search(r"\d+", str(value))
    return match.group(0).zfill(2) if match else str(value)


def find_entry(manifest: Any, number: str) -> dict[str, Any]:
    target = normalize_number(number)
    for entry in notebook_entries(manifest):
        candidates = [
            entry.get("number"),
            entry.get("notebook_number"),
            entry.get("id"),
            entry.get("filename"),
        ]
        if any(
            normalize_number(value) == target
            for value in candidates
            if value is not None
        ):
            return entry
    raise KeyError(f"Notebook {target} is missing from the manifest.")


required_inputs = [
    CONFIG_PATH,
    NOTEBOOK_MANIFEST_PATH,
    NOTEBOOK_03_AUDIT_PATH,
    IMAGING_MANIFEST_PATH,
    IMAGING_INTEGRITY_PATH,
]
missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.exists() or path.stat().st_size == 0
]
if missing_inputs:
    raise FileNotFoundError(
        "Notebook 03 evidence is incomplete:\n"
        + "\n".join(f" - {path}" for path in missing_inputs)
    )

project_config = load_json(CONFIG_PATH)
notebook_manifest = load_json(NOTEBOOK_MANIFEST_PATH)
notebook_03_audit = load_json(NOTEBOOK_03_AUDIT_PATH)
imaging_manifest = load_json(IMAGING_MANIFEST_PATH)
imaging_integrity = load_json(IMAGING_INTEGRITY_PATH)

required_notebook_03_metrics = {
    "prepared_case_success_rate": 1.0,
    "shape_integrity_rate": 1.0,
    "affine_integrity_rate": 1.0,
    "nonempty_reference_mask_rate": 1.0,
    "label_value_validation_rate": 1.0,
    "adapter_reload_success_rate": 1.0,
}
notebook_03_metrics = notebook_03_audit.get("metrics", {})
failed_metrics = [
    key
    for key, expected in required_notebook_03_metrics.items()
    if float(notebook_03_metrics.get(key, 0.0)) != expected
]
if failed_metrics:
    raise RuntimeError(
        "Notebook 03 execution metrics did not pass: " + ", ".join(failed_metrics)
    )
if not imaging_integrity.get("all_cases_passed", False):
    raise RuntimeError("Notebook 03 imaging-integrity report did not pass.")

notebook_03_entry = find_entry(notebook_manifest, "03")
notebook_04_entry = find_entry(notebook_manifest, "04")
notebook_03_status = str(
    notebook_03_entry.get("status", notebook_03_audit.get("status", ""))
).lower()
accepted_notebook_03_statuses = {
    "completed",
    "complete",
    "passed",
    "executed_pending_notebook_save",
}
if notebook_03_status not in accepted_notebook_03_statuses:
    raise RuntimeError(
        f"Notebook 03 is not ready. Current status: {notebook_03_status!r}"
    )

manifest_case_ids = [case.get("case_id") for case in imaging_manifest.get("cases", [])]
if set(manifest_case_ids) != {"stable", "progression", "low-confidence"}:
    raise AssertionError(
        f"Unexpected Notebook 03 case ids: {sorted(manifest_case_ids)}"
    )

NOTEBOOK_FILENAME = notebook_04_entry.get(
    "filename",
    "04_NeuroFHIR_QC_Segmentation_and_Volumetry.ipynb",
)
NOTEBOOK_SAVE_PATH = PROJECT_ROOT / "notebooks" / NOTEBOOK_FILENAME

MODEL_BUNDLE_NAME = "brats_mri_segmentation"
MODEL_BUNDLE_VERSION = "0.5.4"
MODEL_BUNDLE_REVISION = "370f7f9d062745fbac445e7fe6d6616d35df04ec"
MODEL_ROOT = PROJECT_ROOT / "model" / f"{MODEL_BUNDLE_NAME}_v{MODEL_BUNDLE_VERSION}"
MODEL_CHECKPOINT_PATH = MODEL_ROOT / "models/model.pt"
MODEL_PROVENANCE_PATH = MODEL_ROOT / "model_provenance.json"

SEGMENTATION_ROOT = PROJECT_ROOT / "data/sample_masks/notebook_04"
EVALUATION_ROOT = (
    PROJECT_ROOT / "evaluation/results/notebook_04_segmentation_and_volumetry"
)
PREVIEW_ROOT = EVALUATION_ROOT / "previews"
CASE_RESULT_ROOT = EVALUATION_ROOT / "case_results"
SEGMENTATION_MANIFEST_PATH = SEGMENTATION_ROOT / "segmentation_case_manifest.json"
METRICS_JSON_PATH = EVALUATION_ROOT / "segmentation_metrics.json"
METRICS_CSV_PATH = EVALUATION_ROOT / "segmentation_metrics.csv"
VOLUMETRY_CSV_PATH = EVALUATION_ROOT / "volumetry_results.csv"
RUNTIME_LOG_PATH = EVALUATION_ROOT / "inference_runtime_log.json"
AUDIT_JSON_PATH = (
    PROJECT_ROOT / "evaluation/results/notebook_04_segmentation_volumetry_audit.json"
)
AUDIT_MD_PATH = PROJECT_ROOT / "docs/NOTEBOOK_04_SEGMENTATION_AND_VOLUMETRY.md"

for directory in (
    MODEL_ROOT,
    SEGMENTATION_ROOT,
    EVALUATION_ROOT,
    PREVIEW_ROOT,
    CASE_RESULT_ROOT,
):
    directory.mkdir(parents=True, exist_ok=True)

print("=" * 96)
print("✅ Notebook 03 imaging-evidence gate passed")
print(f"✅ Notebook 03 status accepted: {notebook_03_status}")
print(f"✅ Prepared cases: {sorted(manifest_case_ids)}")
print(f"📓 Notebook 04 target path: {NOTEBOOK_SAVE_PATH}")
print("⚠️ The public images are linked to synthetic FHIR demo context only")
print("=" * 96)

Mounted at /content/drive
✅ Notebook 03 imaging-evidence gate passed
✅ Notebook 03 status accepted: executed_pending_notebook_save
✅ Prepared cases: ['low-confidence', 'progression', 'stable']
📓 Notebook 04 target path: /content/drive/MyDrive/neurofhir-qc/notebooks/04_NeuroFHIR_QC_Segmentation_and_Volumetry.ipynb
⚠️ The public images are linked to synthetic FHIR demo context only


In [2]:
# Cell 2 — Install and verify the segmentation runtime

required_packages = [
    "monai==1.6.0",
    "huggingface_hub>=0.30,<2",
    "scipy>=1.11,<2",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", *required_packages]
)

import importlib.metadata as importlib_metadata

import matplotlib
import matplotlib.pyplot as plt
import monai
import nibabel as nib
import numpy as np
import pandas as pd
import scipy
import torch
from huggingface_hub import snapshot_download
from monai.inferers import sliding_window_inference
from monai.networks.nets import SegResNet
from scipy import ndimage

if not torch.cuda.is_available():
    raise RuntimeError(
        "Notebook 04 requires a GPU. In Colab choose Runtime → Change runtime type → GPU, "
        "then restart and run from Cell 1."
    )

DEVICE = torch.device("cuda:0")
GPU_PROPERTIES = torch.cuda.get_device_properties(0)
GPU_MEMORY_GB = GPU_PROPERTIES.total_memory / (1024 ** 3)
USE_AMP = True

if GPU_MEMORY_GB >= 32:
    ROI_SIZE = (240, 240, 160)
elif GPU_MEMORY_GB >= 20:
    ROI_SIZE = (192, 192, 144)
else:
    ROI_SIZE = (160, 160, 128)

SW_BATCH_SIZE = 1
SW_OVERLAP = 0.5
MODEL_THRESHOLD = 0.5

runtime_versions = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "monai": monai.__version__,
    "nibabel": nib.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "matplotlib": matplotlib.__version__,
    "huggingface_hub": importlib_metadata.version("huggingface_hub"),
}

for name, version in runtime_versions.items():
    print(f"{name}: {version}")
print(f"GPU: {GPU_PROPERTIES.name}")
print(f"GPU memory: {GPU_MEMORY_GB:.2f} GiB")
print(f"Sliding-window ROI: {ROI_SIZE}")
print(f"AMP enabled: {USE_AMP}")
print("=" * 96)
print("✅ Segmentation runtime installed and GPU verified")
print("=" * 96)

python: 3.12.13
torch: 2.11.0+cu128
monai: 1.6.0
nibabel: 5.4.2
numpy: 2.0.2
pandas: 2.2.2
scipy: 1.16.3
matplotlib: 3.10.0
huggingface_hub: 1.23.0
GPU: NVIDIA L4
GPU memory: 22.03 GiB
Sliding-window ROI: (192, 192, 144)
AMP enabled: True
✅ Segmentation runtime installed and GPU verified


In [3]:
# Cell 3 — Reload and validate the three four-modality model inputs

MODEL_INPUT_CHANNELS = [
    ("T1c", ("t1gd", "t1c", "t1ce", "t1-contrast", "contrast")),
    ("T1", ("t1w", "t1")),
    ("T2", ("t2w", "t2")),
    ("FLAIR", ("flair",)),
]
CASE_ORDER = ["stable", "progression", "low-confidence"]


def resolve_modality_path(
    modality_files: dict[str, str],
    aliases: tuple[str, ...],
) -> tuple[str, Path]:
    normalized = {key.lower(): key for key in modality_files}
    for alias in aliases:
        if alias in normalized:
            original_key = normalized[alias]
            return original_key, PROJECT_ROOT / modality_files[original_key]
    for original_key, relative_path in modality_files.items():
        lowered = original_key.lower()
        if any(alias in lowered for alias in aliases):
            if "t1" in aliases and any(token in lowered for token in ("t1gd", "t1c", "t1ce")):
                continue
            return original_key, PROJECT_ROOT / relative_path
    raise KeyError(
        f"No modality matching aliases {aliases} in {sorted(modality_files)}"
    )


case_by_id = {
    case["case_id"]: case
    for case in imaging_manifest["cases"]
}
validated_case_inputs: list[dict[str, Any]] = []

for case_id in CASE_ORDER:
    case = case_by_id[case_id]
    modality_files = case.get("modality_files", {})
    if len(modality_files) != 4:
        raise AssertionError(
            f"{case_id} has {len(modality_files)} modalities instead of four."
        )

    ordered_modalities = []
    for model_channel, aliases in MODEL_INPUT_CHANNELS:
        source_name, path = resolve_modality_path(modality_files, aliases)
        if not path.exists() or path.stat().st_size == 0:
            raise FileNotFoundError(f"Missing {case_id} {model_channel}: {path}")
        ordered_modalities.append(
            {
                "model_channel": model_channel,
                "source_modality_name": source_name,
                "path": path,
            }
        )

    reference_multiclass_path = (
        PROJECT_ROOT / case["reference_multiclass_mask_file"]
    )
    reference_wt_path = (
        PROJECT_ROOT / case["reference_whole_tumor_mask_file"]
    )
    for path in (reference_multiclass_path, reference_wt_path):
        if not path.exists() or path.stat().st_size == 0:
            raise FileNotFoundError(f"Missing reference mask: {path}")

    modality_images = [nib.load(str(item["path"])) for item in ordered_modalities]
    shapes = [tuple(int(v) for v in image.shape) for image in modality_images]
    if len(set(shapes)) != 1:
        raise AssertionError(f"{case_id} modality shape mismatch: {shapes}")
    first_affine = modality_images[0].affine
    if not all(
        np.allclose(first_affine, image.affine, atol=1e-4, rtol=0.0)
        for image in modality_images[1:]
    ):
        raise AssertionError(f"{case_id} modality affine mismatch.")

    reference_image = nib.load(str(reference_multiclass_path))
    reference_data = np.asanyarray(reference_image.dataobj)
    if tuple(reference_data.shape) != shapes[0]:
        raise AssertionError(f"{case_id} image/reference shape mismatch.")
    if not np.allclose(first_affine, reference_image.affine, atol=1e-4, rtol=0.0):
        raise AssertionError(f"{case_id} image/reference affine mismatch.")

    reference_labels = sorted(int(v) for v in np.unique(np.rint(reference_data)))
    if set(reference_labels) - {0, 1, 2, 3}:
        raise AssertionError(
            f"{case_id} contains unexpected reference labels: {reference_labels}"
        )

    spacing_mm = tuple(float(v) for v in reference_image.header.get_zooms()[:3])
    validated_case_inputs.append(
        {
            "case_id": case_id,
            "source_case_id": case["source_case_id"],
            "patient_reference": case["patient_reference"],
            "followup_imaging_reference": case["followup_imaging_reference"],
            "ordered_modalities": ordered_modalities,
            "reference_multiclass_path": reference_multiclass_path,
            "reference_whole_tumor_path": reference_wt_path,
            "shape": shapes[0],
            "spacing_mm": spacing_mm,
            "affine": first_affine,
            "planned_future_perturbation": bool(
                case.get("planned_future_perturbation", False)
            ),
        }
    )

print("=" * 96)
print("✅ Three model-input packages validated")
for case in validated_case_inputs:
    channel_summary = ", ".join(
        f"{item['model_channel']}←{item['source_modality_name']}"
        for item in case["ordered_modalities"]
    )
    print(
        f" - {case['case_id']}: {case['source_case_id']} | "
        f"shape={case['shape']} | spacing={case['spacing_mm']} | {channel_summary}"
    )
print("✅ Model input order is T1c, T1, T2, FLAIR")
print("=" * 96)

✅ Three model-input packages validated
 - stable: BRATS_029 | shape=(240, 240, 155) | spacing=(1.0, 1.0, 1.0) | T1c←t1gd, T1←T1w, T2←T2w, FLAIR←FLAIR
 - progression: BRATS_167 | shape=(240, 240, 155) | spacing=(1.0, 1.0, 1.0) | T1c←t1gd, T1←T1w, T2←T2w, FLAIR←FLAIR
 - low-confidence: BRATS_443 | shape=(240, 240, 155) | spacing=(1.0, 1.0, 1.0) | T1c←t1gd, T1←T1w, T2←T2w, FLAIR←FLAIR
✅ Model input order is T1c, T1, T2, FLAIR


In [4]:
# Cell 4 — Obtain and verify the pinned MONAI BraTS model artifact

MODEL_REPO_ID = "MONAI/brats_mri_segmentation"
MODEL_EXPECTED_SHA256 = (
    "860ccb3f1c21c99d0410ad8a1ac4ef6b8fab60cec0a503b0ba42675741a750ae"
)
MODEL_PINNED_URL = (
    "https://huggingface.co/MONAI/brats_mri_segmentation/resolve/"
    f"{MODEL_BUNDLE_REVISION}/models/model.pt?download=true"
)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)


def checkpoint_is_verified(path: Path) -> bool:
    return (
        path.exists()
        and path.is_file()
        and path.stat().st_size > 1024 * 1024
        and sha256_file(path) == MODEL_EXPECTED_SHA256
    )


model_download_source = "existing-verified-artifact"
download_error = None

if not checkpoint_is_verified(MODEL_CHECKPOINT_PATH):
    if MODEL_CHECKPOINT_PATH.exists():
        MODEL_CHECKPOINT_PATH.unlink()

    try:
        snapshot_download(
            repo_id=MODEL_REPO_ID,
            revision=MODEL_BUNDLE_REVISION,
            local_dir=str(MODEL_ROOT),
            allow_patterns=[
                "models/model.pt",
                "configs/metadata.json",
                "configs/inference.json",
                "docs/README.md",
                "LICENSE",
            ],
            max_workers=4,
        )
        model_download_source = "huggingface-pinned-commit"
    except Exception as exc:
        download_error = f"{type(exc).__name__}: {exc}"
        print(
            "Pinned Hugging Face snapshot download failed; "
            "trying the exact pinned model URL."
        )
        from urllib.request import urlretrieve

        MODEL_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(MODEL_PINNED_URL, MODEL_CHECKPOINT_PATH)
        model_download_source = "huggingface-pinned-url-fallback"

if not checkpoint_is_verified(MODEL_CHECKPOINT_PATH):
    actual_sha256 = (
        sha256_file(MODEL_CHECKPOINT_PATH)
        if MODEL_CHECKPOINT_PATH.exists()
        else None
    )
    raise RuntimeError(
        "The model checkpoint failed pinned SHA-256 verification. "
        f"Expected {MODEL_EXPECTED_SHA256}; found {actual_sha256}."
    )

model_provenance = {
    "project_name": project_config["project_name"],
    "generated_utc": utc_now(),
    "bundle_name": MODEL_BUNDLE_NAME,
    "bundle_version": MODEL_BUNDLE_VERSION,
    "bundle_revision": MODEL_BUNDLE_REVISION,
    "repository": MODEL_REPO_ID,
    "download_source": model_download_source,
    "huggingface_download_error_if_any": download_error,
    "checkpoint_relative_path": (
        MODEL_CHECKPOINT_PATH.relative_to(PROJECT_ROOT).as_posix()
    ),
    "checkpoint_size_bytes": MODEL_CHECKPOINT_PATH.stat().st_size,
    "checkpoint_sha256": sha256_file(MODEL_CHECKPOINT_PATH),
    "architecture": {
        "name": "MONAI SegResNet",
        "spatial_dims": 3,
        "in_channels": 4,
        "out_channels": 3,
        "init_filters": 16,
        "blocks_down": [1, 2, 2, 4],
        "blocks_up": [1, 1, 1],
        "dropout_prob": 0.2,
    },
    "input_channel_order": ["T1c", "T1", "T2", "FLAIR"],
    "output_channel_order": [
        "tumor_core",
        "whole_tumor",
        "enhancing_tumor",
    ],
    "threshold": MODEL_THRESHOLD,
    "intended_use_boundary": (
        "Research/example use only; not a diagnostic device."
    ),
    "evaluation_boundary": (
        "The three prepared MSD cases are a demonstration benchmark and "
        "are not claimed as independent external validation."
    ),
}
write_json(MODEL_PROVENANCE_PATH, model_provenance)

print("=" * 96)
print("✅ Pinned MONAI BraTS model artifact verified")
print(f"📦 Bundle version: {MODEL_BUNDLE_VERSION}")
print(f"📌 Pinned commit: {MODEL_BUNDLE_REVISION}")
print(f"📥 Source: {model_download_source}")
print(f"📄 Checkpoint: {MODEL_CHECKPOINT_PATH}")
print(f"🔐 SHA-256: {MODEL_EXPECTED_SHA256}")
print("=" * 96)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Pinned MONAI BraTS model artifact verified
📦 Bundle version: 0.5.4
📌 Pinned commit: 370f7f9d062745fbac445e7fe6d6616d35df04ec
📥 Source: huggingface-pinned-commit
📄 Checkpoint: /content/drive/MyDrive/neurofhir-qc/model/brats_mri_segmentation_v0.5.4/models/model.pt
🔐 SHA-256: 860ccb3f1c21c99d0410ad8a1ac4ef6b8fab60cec0a503b0ba42675741a750ae


In [5]:
# Cell 5 — Define model loading, preprocessing, inference, postprocessing, and metrics

np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


def extract_state_dict(checkpoint: Any) -> dict[str, torch.Tensor]:
    if isinstance(checkpoint, dict):
        for key in ("model", "state_dict", "network"):
            candidate = checkpoint.get(key)
            if isinstance(candidate, dict) and candidate:
                return candidate
        if checkpoint and all(torch.is_tensor(value) for value in checkpoint.values()):
            return checkpoint
    raise TypeError("Could not identify a model state dictionary in the checkpoint.")


def clean_state_dict(state: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    cleaned = {}
    for key, value in state.items():
        clean_key = key
        for prefix in ("module.", "_orig_mod."):
            if clean_key.startswith(prefix):
                clean_key = clean_key[len(prefix):]
        cleaned[clean_key] = value
    return cleaned


network = SegResNet(
    spatial_dims=3,
    init_filters=16,
    in_channels=4,
    out_channels=3,
    dropout_prob=0.2,
    blocks_down=(1, 2, 2, 4),
    blocks_up=(1, 1, 1),
)

try:
    checkpoint = torch.load(
        MODEL_CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=True,
    )
except TypeError:
    checkpoint = torch.load(MODEL_CHECKPOINT_PATH, map_location="cpu")
except Exception as exc:
    print(
        "weights_only checkpoint loading was unavailable for this trusted pinned artifact; "
        "retrying standard loading."
    )
    print(f"Reason: {type(exc).__name__}: {exc}")
    checkpoint = torch.load(
        MODEL_CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=False,
    )

state_dict = clean_state_dict(extract_state_dict(checkpoint))
load_result = network.load_state_dict(state_dict, strict=True)
if load_result.missing_keys or load_result.unexpected_keys:
    raise RuntimeError(
        f"Checkpoint mismatch: missing={load_result.missing_keys}, "
        f"unexpected={load_result.unexpected_keys}"
    )
network = network.to(DEVICE)
network.eval()
del checkpoint, state_dict


def normalize_nonzero_channels(channels: np.ndarray) -> np.ndarray:
    if channels.ndim != 4 or channels.shape[0] != 4:
        raise ValueError(f"Expected [4, X, Y, Z] input; found {channels.shape}")
    normalized = np.zeros_like(channels, dtype=np.float32)
    for index in range(channels.shape[0]):
        channel = channels[index].astype(np.float32, copy=False)
        mask = np.isfinite(channel) & (channel != 0)
        if not mask.any():
            raise ValueError(f"Input channel {index} has no non-zero finite voxels.")
        values = channel[mask]
        mean = float(values.mean())
        std = float(values.std())
        if std <= 1e-8:
            raise ValueError(f"Input channel {index} has near-zero standard deviation.")
        normalized[index, mask] = (values - mean) / std
    return normalized


def run_model(channels: np.ndarray) -> tuple[np.ndarray, float]:
    normalized = normalize_nonzero_channels(channels)
    tensor = torch.from_numpy(normalized[None]).to(DEVICE, non_blocking=True)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            logits = sliding_window_inference(
                inputs=tensor,
                roi_size=ROI_SIZE,
                sw_batch_size=SW_BATCH_SIZE,
                predictor=network,
                overlap=SW_OVERLAP,
                mode="gaussian",
            )
        probabilities = torch.sigmoid(logits).float()
    torch.cuda.synchronize()
    elapsed_seconds = time.perf_counter() - started
    output = probabilities[0].cpu().numpy().astype(np.float32, copy=False)
    del tensor, logits, probabilities
    torch.cuda.empty_cache()
    if output.shape[0] != 3:
        raise AssertionError(f"Expected three output channels; found {output.shape}")
    return output, elapsed_seconds


def enforce_nested_regions(
    probabilities: np.ndarray,
    threshold: float,
) -> dict[str, np.ndarray]:
    tumor_core_raw = probabilities[0] >= threshold
    whole_tumor_raw = probabilities[1] >= threshold
    enhancing_raw = probabilities[2] >= threshold

    enhancing = enhancing_raw
    tumor_core = tumor_core_raw | enhancing
    whole_tumor = whole_tumor_raw | tumor_core

    # MSD Task01 uses 1=non-enhancing/necrotic core, 2=edema, 3=enhancing tumor.
    multiclass = np.zeros(whole_tumor.shape, dtype=np.uint8)
    multiclass[whole_tumor] = 2
    multiclass[tumor_core] = 1
    multiclass[enhancing] = 3

    return {
        "tumor_core": tumor_core.astype(np.uint8),
        "whole_tumor": whole_tumor.astype(np.uint8),
        "enhancing_tumor": enhancing.astype(np.uint8),
        "multiclass_msd": multiclass,
        "hierarchy_added_tc_voxels": int(np.count_nonzero(tumor_core & ~tumor_core_raw)),
        "hierarchy_added_wt_voxels": int(np.count_nonzero(whole_tumor & ~whole_tumor_raw)),
    }


def save_nifti_like(
    data: np.ndarray,
    reference_image: nib.Nifti1Image,
    destination: Path,
    dtype: np.dtype,
) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    header = reference_image.header.copy()
    header.set_data_dtype(dtype)
    output = nib.Nifti1Image(
        data.astype(dtype, copy=False),
        reference_image.affine,
        header,
    )
    nib.save(output, str(destination))
    if not destination.exists() or destination.stat().st_size == 0:
        raise IOError(f"Failed to save NIfTI output: {destination}")


def binary_overlap_metrics(
    prediction: np.ndarray,
    reference: np.ndarray,
) -> dict[str, float | int | None]:
    prediction = prediction.astype(bool)
    reference = reference.astype(bool)
    tp = int(np.count_nonzero(prediction & reference))
    fp = int(np.count_nonzero(prediction & ~reference))
    fn = int(np.count_nonzero(~prediction & reference))
    denominator = 2 * tp + fp + fn
    dice = 1.0 if denominator == 0 else (2 * tp) / denominator
    precision = None if tp + fp == 0 else tp / (tp + fp)
    sensitivity = None if tp + fn == 0 else tp / (tp + fn)
    return {
        "true_positive_voxels": tp,
        "false_positive_voxels": fp,
        "false_negative_voxels": fn,
        "dice": float(dice),
        "precision": None if precision is None else float(precision),
        "sensitivity": None if sensitivity is None else float(sensitivity),
    }


def hd95_mm(
    prediction: np.ndarray,
    reference: np.ndarray,
    spacing_mm: tuple[float, float, float],
) -> float | None:
    prediction = prediction.astype(bool)
    reference = reference.astype(bool)
    if not prediction.any() and not reference.any():
        return 0.0
    if not prediction.any() or not reference.any():
        return None

    structure = ndimage.generate_binary_structure(3, 1)
    pred_surface = prediction ^ ndimage.binary_erosion(
        prediction, structure=structure, border_value=0
    )
    ref_surface = reference ^ ndimage.binary_erosion(
        reference, structure=structure, border_value=0
    )
    if not pred_surface.any():
        pred_surface = prediction
    if not ref_surface.any():
        ref_surface = reference

    distance_to_ref = ndimage.distance_transform_edt(
        ~ref_surface, sampling=spacing_mm
    )
    distance_to_pred = ndimage.distance_transform_edt(
        ~pred_surface, sampling=spacing_mm
    )
    distances = np.concatenate(
        [distance_to_ref[pred_surface], distance_to_pred[ref_surface]]
    )
    return float(np.percentile(distances, 95))


def volume_ml(mask: np.ndarray, spacing_mm: tuple[float, float, float]) -> float:
    voxel_volume_mm3 = float(np.prod(spacing_mm))
    return float(np.count_nonzero(mask) * voxel_volume_mm3 / 1000.0)


def round_optional(value: float | None, digits: int = 6) -> float | None:
    return None if value is None else round(float(value), digits)


print("=" * 96)
print("✅ MONAI SegResNet checkpoint loaded with strict state matching")
print("✅ Preprocessing, sliding-window inference, hierarchy, volume, and metric functions defined")
print("=" * 96)

✅ MONAI SegResNet checkpoint loaded with strict state matching
✅ Preprocessing, sliding-window inference, hierarchy, volume, and metric functions defined


In [6]:
# Cell 6 — Run baseline segmentation and volumetry for all three cases

# Remove only Notebook 04 generated case outputs so reruns cannot mix stale masks.
for case_id in CASE_ORDER:
    case_output_root = SEGMENTATION_ROOT / case_id
    if case_output_root.exists():
        shutil.rmtree(case_output_root)
    case_output_root.mkdir(parents=True, exist_ok=True)

case_results: list[dict[str, Any]] = []
runtime_rows: list[dict[str, Any]] = []

for position, case in enumerate(validated_case_inputs, start=1):
    case_id = case["case_id"]
    print(f"[{position}/{len(validated_case_inputs)}] Loading {case_id} ...")

    modality_arrays = []
    modality_sha256 = {}
    for item in case["ordered_modalities"]:
        image = nib.load(str(item["path"]))
        array = np.asanyarray(image.dataobj).astype(np.float32, copy=False)
        if not np.isfinite(array).all():
            raise ValueError(f"Non-finite MRI values in {item['path']}")
        modality_arrays.append(array)
        modality_sha256[item["model_channel"]] = sha256_file(item["path"])
    channels = np.stack(modality_arrays, axis=0)

    reference_image = nib.load(str(case["reference_multiclass_path"]))
    reference_multiclass = np.rint(
        np.asanyarray(reference_image.dataobj)
    ).astype(np.uint8)
    reference_regions = {
        "tumor_core": np.isin(reference_multiclass, [1, 3]),
        "whole_tumor": reference_multiclass > 0,
        "enhancing_tumor": reference_multiclass == 3,
    }

    probabilities, elapsed_seconds = run_model(channels)
    if probabilities.shape[1:] != reference_multiclass.shape:
        raise AssertionError(
            f"{case_id} output shape {probabilities.shape[1:]} does not match "
            f"reference {reference_multiclass.shape}."
        )
    if not np.isfinite(probabilities).all():
        raise ValueError(f"{case_id} model probabilities contain non-finite values.")
    if probabilities.min() < 0.0 or probabilities.max() > 1.0:
        raise AssertionError(f"{case_id} probabilities are outside [0, 1].")

    prediction = enforce_nested_regions(probabilities, MODEL_THRESHOLD)
    if int(np.count_nonzero(prediction["whole_tumor"])) <= 0:
        raise AssertionError(f"{case_id} produced an empty whole-tumor mask.")

    case_output_root = SEGMENTATION_ROOT / case_id
    output_paths = {
        "predicted_tumor_core_mask_file": case_output_root / "tumor_core_binary.nii.gz",
        "predicted_whole_tumor_mask_file": case_output_root / "whole_tumor_binary.nii.gz",
        "predicted_enhancing_tumor_mask_file": case_output_root / "enhancing_tumor_binary.nii.gz",
        "predicted_multiclass_mask_file": case_output_root / "predicted_multiclass_msd_labels.nii.gz",
        "whole_tumor_probability_file": case_output_root / "whole_tumor_probability_float16.npz",
    }
    save_nifti_like(
        prediction["tumor_core"], reference_image,
        output_paths["predicted_tumor_core_mask_file"], np.uint8,
    )
    save_nifti_like(
        prediction["whole_tumor"], reference_image,
        output_paths["predicted_whole_tumor_mask_file"], np.uint8,
    )
    save_nifti_like(
        prediction["enhancing_tumor"], reference_image,
        output_paths["predicted_enhancing_tumor_mask_file"], np.uint8,
    )
    save_nifti_like(
        prediction["multiclass_msd"], reference_image,
        output_paths["predicted_multiclass_mask_file"], np.uint8,
    )
    np.savez_compressed(
        output_paths["whole_tumor_probability_file"],
        probability=probabilities[1].astype(np.float16),
    )

    region_metrics = {}
    for region_name in ("tumor_core", "whole_tumor", "enhancing_tumor"):
        predicted_region = prediction[region_name].astype(bool)
        reference_region = reference_regions[region_name]
        overlap = binary_overlap_metrics(predicted_region, reference_region)
        hd95 = hd95_mm(predicted_region, reference_region, case["spacing_mm"])
        reference_volume = volume_ml(reference_region, case["spacing_mm"])
        predicted_volume = volume_ml(predicted_region, case["spacing_mm"])
        absolute_volume_error = abs(predicted_volume - reference_volume)
        relative_volume_error = (
            None
            if reference_volume <= 0
            else absolute_volume_error / reference_volume
        )
        region_metrics[region_name] = {
            **{
                key: round_optional(value) if isinstance(value, float) else value
                for key, value in overlap.items()
            },
            "hd95_mm": round_optional(hd95),
            "reference_volume_ml": round(reference_volume, 6),
            "predicted_volume_ml": round(predicted_volume, 6),
            "absolute_volume_error_ml": round(absolute_volume_error, 6),
            "relative_volume_error": round_optional(relative_volume_error),
        }

    result = {
        "case_id": case_id,
        "source_case_id": case["source_case_id"],
        "patient_reference": case["patient_reference"],
        "followup_imaging_reference": case["followup_imaging_reference"],
        "generated_utc": utc_now(),
        "model_bundle_name": MODEL_BUNDLE_NAME,
        "model_bundle_version": MODEL_BUNDLE_VERSION,
        "model_bundle_revision": MODEL_BUNDLE_REVISION,
        "model_checkpoint_sha256": model_provenance["checkpoint_sha256"],
        "model_input_order": [item["model_channel"] for item in case["ordered_modalities"]],
        "source_modality_names": [
            item["source_modality_name"] for item in case["ordered_modalities"]
        ],
        "source_modality_sha256": modality_sha256,
        "spatial_shape": list(case["shape"]),
        "spacing_mm": [round(v, 6) for v in case["spacing_mm"]],
        "threshold": MODEL_THRESHOLD,
        "roi_size": list(ROI_SIZE),
        "sliding_window_overlap": SW_OVERLAP,
        "amp_enabled": USE_AMP,
        "inference_seconds": round(elapsed_seconds, 6),
        "hierarchy_postprocessing": {
            "description": (
                "Engineering postprocessing enforces ET subset of TC and TC subset of WT."
            ),
            "added_tumor_core_voxels": prediction["hierarchy_added_tc_voxels"],
            "added_whole_tumor_voxels": prediction["hierarchy_added_wt_voxels"],
        },
        "region_metrics": region_metrics,
        "outputs": {
            key: path.relative_to(PROJECT_ROOT).as_posix()
            for key, path in output_paths.items()
        },
        "reference_multiclass_mask_file": (
            case["reference_multiclass_path"].relative_to(PROJECT_ROOT).as_posix()
        ),
        "planned_future_perturbation": case["planned_future_perturbation"],
        "interpretation": (
            "Baseline model result for a public de-identified research image mapped to "
            "synthetic FHIR demonstration context; not a clinical result."
        ),
    }

    case_result_path = CASE_RESULT_ROOT / f"{case_id}_segmentation_result.json"
    write_json(case_result_path, result)
    result["case_result_file"] = case_result_path.relative_to(PROJECT_ROOT).as_posix()
    case_results.append(result)
    runtime_rows.append(
        {
            "case_id": case_id,
            "source_case_id": case["source_case_id"],
            "gpu_name": GPU_PROPERTIES.name,
            "gpu_memory_gib": round(GPU_MEMORY_GB, 3),
            "roi_size": list(ROI_SIZE),
            "amp_enabled": USE_AMP,
            "inference_seconds": round(elapsed_seconds, 6),
        }
    )

    wt = region_metrics["whole_tumor"]
    print(
        f"    WT Dice={wt['dice']:.4f} | "
        f"predicted={wt['predicted_volume_ml']:.2f} mL | "
        f"reference={wt['reference_volume_ml']:.2f} mL | "
        f"time={elapsed_seconds:.2f}s"
    )
    del channels, modality_arrays, probabilities, reference_multiclass, prediction
    torch.cuda.empty_cache()

segmentation_manifest = {
    "project_name": project_config["project_name"],
    "project_version": project_config.get("version", "0.1.0"),
    "generated_utc": utc_now(),
    "notebook_number": "04",
    "model": model_provenance,
    "data_governance": {
        "public_deidentified_imaging_only": True,
        "synthetic_fhir_context_only": True,
        "real_patient_linkage_claimed": False,
        "clinical_validation_claimed": False,
        "diagnostic_use": False,
    },
    "cases": case_results,
}
write_json(SEGMENTATION_MANIFEST_PATH, segmentation_manifest)
write_json(RUNTIME_LOG_PATH, {"generated_utc": utc_now(), "cases": runtime_rows})

print("=" * 96)
print("✅ Baseline segmentation completed for 3/3 cases")
print(f"📋 Segmentation manifest: {SEGMENTATION_MANIFEST_PATH}")
print("⚠️ No final QC class, human review, or FHIR AI-result write-back occurred")
print("=" * 96)

[1/3] Loading stable ...
    WT Dice=0.8695 | predicted=19.19 mL | reference=18.89 mL | time=2.54s
[2/3] Loading progression ...
    WT Dice=0.9169 | predicted=20.46 mL | reference=22.58 mL | time=1.40s
[3/3] Loading low-confidence ...
    WT Dice=0.9179 | predicted=20.46 mL | reference=21.98 mL | time=1.40s
✅ Baseline segmentation completed for 3/3 cases
📋 Segmentation manifest: /content/drive/MyDrive/neurofhir-qc/data/sample_masks/notebook_04/segmentation_case_manifest.json
⚠️ No final QC class, human review, or FHIR AI-result write-back occurred


In [7]:
# Cell 7 — Create evaluation tables, volumetry summaries, and visual overlays

metric_rows: list[dict[str, Any]] = []
volumetry_rows: list[dict[str, Any]] = []
preview_paths: list[Path] = []

for result in case_results:
    case_id = result["case_id"]
    source_case_id = result["source_case_id"]
    for region_name, metrics in result["region_metrics"].items():
        metric_rows.append(
            {
                "case_id": case_id,
                "source_case_id": source_case_id,
                "region": region_name,
                "dice": metrics["dice"],
                "sensitivity": metrics["sensitivity"],
                "precision": metrics["precision"],
                "hd95_mm": metrics["hd95_mm"],
                "reference_volume_ml": metrics["reference_volume_ml"],
                "predicted_volume_ml": metrics["predicted_volume_ml"],
                "absolute_volume_error_ml": metrics["absolute_volume_error_ml"],
                "relative_volume_error": metrics["relative_volume_error"],
                "inference_seconds": result["inference_seconds"],
            }
        )

    wt = result["region_metrics"]["whole_tumor"]
    volumetry_rows.append(
        {
            "case_id": case_id,
            "source_case_id": source_case_id,
            "patient_reference": result["patient_reference"],
            "followup_imaging_reference": result["followup_imaging_reference"],
            "reference_whole_tumor_volume_ml": wt["reference_volume_ml"],
            "predicted_whole_tumor_volume_ml": wt["predicted_volume_ml"],
            "absolute_volume_error_ml": wt["absolute_volume_error_ml"],
            "relative_volume_error": wt["relative_volume_error"],
            "whole_tumor_dice": wt["dice"],
            "whole_tumor_hd95_mm": wt["hd95_mm"],
            "inference_seconds": result["inference_seconds"],
        }
    )

    case = next(item for item in validated_case_inputs if item["case_id"] == case_id)
    flair_item = next(
        item for item in case["ordered_modalities"] if item["model_channel"] == "FLAIR"
    )
    flair = np.asanyarray(nib.load(str(flair_item["path"])).dataobj)
    reference = np.asanyarray(
        nib.load(str(case["reference_whole_tumor_path"])).dataobj
    ) > 0
    predicted_path = PROJECT_ROOT / result["outputs"]["predicted_whole_tumor_mask_file"]
    predicted = np.asanyarray(nib.load(str(predicted_path)).dataobj) > 0
    union = reference | predicted
    slice_index = int(np.argmax(union.sum(axis=(0, 1))))

    image_slice = flair[:, :, slice_index].astype(np.float32)
    nonzero = image_slice[np.isfinite(image_slice) & (image_slice != 0)]
    if nonzero.size:
        lower, upper = np.percentile(nonzero, [1.0, 99.0])
        if upper <= lower:
            upper = lower + 1.0
        image_slice = np.clip((image_slice - lower) / (upper - lower), 0.0, 1.0)
    else:
        image_slice = np.zeros_like(image_slice)

    reference_slice = reference[:, :, slice_index]
    predicted_slice = predicted[:, :, slice_index]
    false_positive = predicted_slice & ~reference_slice
    false_negative = reference_slice & ~predicted_slice

    preview_path = PREVIEW_ROOT / f"{case_id}_segmentation_evaluation.png"
    figure, axes = plt.subplots(1, 4, figsize=(18, 5))
    axes[0].imshow(np.rot90(image_slice), cmap="gray")
    axes[0].set_title(f"{case_id}: FLAIR\nslice {slice_index}")
    axes[0].axis("off")

    axes[1].imshow(np.rot90(image_slice), cmap="gray")
    axes[1].imshow(
        np.rot90(reference_slice.astype(np.float32)),
        cmap="autumn", alpha=np.rot90(reference_slice.astype(np.float32)) * 0.5,
        vmin=0, vmax=1,
    )
    axes[1].set_title("Expert WT reference")
    axes[1].axis("off")

    axes[2].imshow(np.rot90(image_slice), cmap="gray")
    axes[2].imshow(
        np.rot90(predicted_slice.astype(np.float32)),
        cmap="winter", alpha=np.rot90(predicted_slice.astype(np.float32)) * 0.5,
        vmin=0, vmax=1,
    )
    axes[2].set_title(
        f"Predicted WT\nDice {result['region_metrics']['whole_tumor']['dice']:.3f}"
    )
    axes[2].axis("off")

    error_map = np.zeros((*false_positive.shape, 3), dtype=np.float32)
    error_map[false_positive] = [1.0, 0.0, 0.0]
    error_map[false_negative] = [0.0, 0.0, 1.0]
    alpha = (false_positive | false_negative).astype(np.float32) * 0.7
    axes[3].imshow(np.rot90(image_slice), cmap="gray")
    axes[3].imshow(np.rot90(error_map), alpha=np.rot90(alpha))
    axes[3].set_title("Error: red FP / blue FN")
    axes[3].axis("off")

    figure.tight_layout()
    figure.savefig(preview_path, dpi=170, bbox_inches="tight")
    plt.close(figure)
    if not preview_path.exists() or preview_path.stat().st_size == 0:
        raise IOError(f"Preview was not created: {preview_path}")
    preview_paths.append(preview_path)

metrics_dataframe = pd.DataFrame(metric_rows)
volumetry_dataframe = pd.DataFrame(volumetry_rows)
metrics_dataframe.to_csv(METRICS_CSV_PATH, index=False)
volumetry_dataframe.to_csv(VOLUMETRY_CSV_PATH, index=False)

whole_tumor_rows = metrics_dataframe[
    metrics_dataframe["region"] == "whole_tumor"
].copy()
summary_metrics = {
    "case_count": len(case_results),
    "inference_success_rate": len(case_results) / 3,
    "mean_whole_tumor_dice": round(float(whole_tumor_rows["dice"].mean()), 6),
    "median_whole_tumor_dice": round(float(whole_tumor_rows["dice"].median()), 6),
    "mean_whole_tumor_hd95_mm": round(
        float(whole_tumor_rows["hd95_mm"].dropna().mean()), 6
    ),
    "mean_absolute_whole_tumor_volume_error_ml": round(
        float(whole_tumor_rows["absolute_volume_error_ml"].mean()), 6
    ),
    "mean_relative_whole_tumor_volume_error": round(
        float(whole_tumor_rows["relative_volume_error"].mean()), 6
    ),
    "mean_inference_seconds": round(
        float(volumetry_dataframe["inference_seconds"].mean()), 6
    ),
    "model_output_nonempty_rate": 1.0,
    "output_shape_integrity_rate": 1.0,
    "output_affine_integrity_rate": 1.0,
    "preview_count": len(preview_paths),
}

write_json(
    METRICS_JSON_PATH,
    {
        "project_name": project_config["project_name"],
        "generated_utc": utc_now(),
        "model_bundle_name": MODEL_BUNDLE_NAME,
        "model_bundle_version": MODEL_BUNDLE_VERSION,
        "model_bundle_revision": MODEL_BUNDLE_REVISION,
        "evaluation_interpretation": (
            "Three-case executable demonstration benchmark; not independent external validation."
        ),
        "summary": summary_metrics,
        "rows": metric_rows,
    },
)

print("=" * 96)
print("✅ Segmentation metrics and volumetry tables created")
print(f"✅ Evaluation previews: {len(preview_paths)}/3")
print(f"📊 Mean whole-tumor Dice: {summary_metrics['mean_whole_tumor_dice']:.4f}")
print(
    "📏 Mean absolute whole-tumor volume error: "
    f"{summary_metrics['mean_absolute_whole_tumor_volume_error_ml']:.3f} mL"
)
print(f"⏱️ Mean inference time: {summary_metrics['mean_inference_seconds']:.2f} s")
print("⚠️ These results are demonstration metrics, not clinical or independent external validation")
print("=" * 96)

✅ Segmentation metrics and volumetry tables created
✅ Evaluation previews: 3/3
📊 Mean whole-tumor Dice: 0.9014
📏 Mean absolute whole-tumor volume error: 1.313 mL
⏱️ Mean inference time: 1.78 s
⚠️ These results are demonstration metrics, not clinical or independent external validation


In [8]:
# Cell 8 — Create reusable segmentation code, verification script, requirements, and model card

SEGMENTATION_SERVICE_PATH = (
    PROJECT_ROOT / "backend/app/services/segmentation_service.py"
)
VERIFY_SCRIPT_PATH = PROJECT_ROOT / "scripts/verify_segmentation_results.py"
REQUIREMENTS_PATH = PROJECT_ROOT / "requirements/segmentation.txt"
MODEL_CARD_PATH = PROJECT_ROOT / "model/model_card.md"
TECHNICAL_DOC_PATH = PROJECT_ROOT / "docs/SEGMENTATION_AND_VOLUMETRY.md"

segmentation_service_source = r'''# Reusable MONAI inference helpers for NeuroFHIR-QC.
# Research demonstration only. This module does not perform FHIR write-back,
# human review, or clinical decision-making.

from __future__ import annotations

from pathlib import Path
from typing import Any

import nibabel as nib
import numpy as np
import torch
from monai.inferers import sliding_window_inference
from monai.networks.nets import SegResNet

MODEL_INPUT_ORDER = ("T1c", "T1", "T2", "FLAIR")
OUTPUT_CHANNELS = {"tumor_core": 0, "whole_tumor": 1, "enhancing_tumor": 2}


def _state_dict(checkpoint: Any) -> dict[str, torch.Tensor]:
    if isinstance(checkpoint, dict):
        for key in ("model", "state_dict", "network"):
            candidate = checkpoint.get(key)
            if isinstance(candidate, dict) and candidate:
                return candidate
        if checkpoint and all(torch.is_tensor(value) for value in checkpoint.values()):
            return checkpoint
    raise TypeError("Checkpoint does not contain a recognizable state dictionary.")


def load_brats_model(checkpoint_path: Path, device: torch.device) -> SegResNet:
    model = SegResNet(
        spatial_dims=3,
        init_filters=16,
        in_channels=4,
        out_channels=3,
        dropout_prob=0.2,
        blocks_down=(1, 2, 2, 4),
        blocks_up=(1, 1, 1),
    )
    try:
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    except TypeError:
        checkpoint = torch.load(checkpoint_path, map_location="cpu")
    state = {}
    for key, value in _state_dict(checkpoint).items():
        for prefix in ("module.", "_orig_mod."):
            if key.startswith(prefix):
                key = key[len(prefix):]
        state[key] = value
    model.load_state_dict(state, strict=True)
    return model.to(device).eval()


def normalize_nonzero_channels(channels: np.ndarray) -> np.ndarray:
    if channels.ndim != 4 or channels.shape[0] != 4:
        raise ValueError(f"Expected [4, X, Y, Z], found {channels.shape}")
    output = np.zeros_like(channels, dtype=np.float32)
    for index, channel in enumerate(channels):
        mask = np.isfinite(channel) & (channel != 0)
        if not mask.any():
            raise ValueError(f"Channel {index} contains no usable non-zero voxels.")
        values = channel[mask].astype(np.float32)
        std = float(values.std())
        if std <= 1e-8:
            raise ValueError(f"Channel {index} has near-zero standard deviation.")
        output[index, mask] = (values - float(values.mean())) / std
    return output


def infer_probabilities(
    model: torch.nn.Module,
    channels: np.ndarray,
    device: torch.device,
    roi_size: tuple[int, int, int],
    overlap: float = 0.5,
    amp: bool = True,
) -> np.ndarray:
    tensor = torch.from_numpy(normalize_nonzero_channels(channels)[None]).to(device)
    with torch.inference_mode(), torch.autocast(
        device_type=device.type,
        dtype=torch.float16,
        enabled=amp and device.type == "cuda",
    ):
        logits = sliding_window_inference(
            tensor, roi_size, 1, model, overlap=overlap, mode="gaussian"
        )
    return torch.sigmoid(logits)[0].float().cpu().numpy()


def enforce_nested_regions(
    probabilities: np.ndarray,
    threshold: float = 0.5,
) -> dict[str, np.ndarray]:
    tc = probabilities[0] >= threshold
    wt = probabilities[1] >= threshold
    et = probabilities[2] >= threshold
    tc = tc | et
    wt = wt | tc
    # MSD Task01 labels: 1=core, 2=edema, 3=enhancing tumor.
    multiclass = np.zeros(wt.shape, dtype=np.uint8)
    multiclass[wt] = 2
    multiclass[tc] = 1
    multiclass[et] = 3
    return {
        "tumor_core": tc.astype(np.uint8),
        "whole_tumor": wt.astype(np.uint8),
        "enhancing_tumor": et.astype(np.uint8),
        "multiclass_msd": multiclass,
    }


def mask_volume_ml(mask: np.ndarray, spacing_mm: tuple[float, float, float]) -> float:
    return float(np.count_nonzero(mask) * np.prod(spacing_mm) / 1000.0)


def save_mask_like(
    mask: np.ndarray,
    reference_path: Path,
    destination: Path,
) -> None:
    reference = nib.load(str(reference_path))
    destination.parent.mkdir(parents=True, exist_ok=True)
    header = reference.header.copy()
    header.set_data_dtype(np.uint8)
    nib.save(
        nib.Nifti1Image(mask.astype(np.uint8), reference.affine, header),
        str(destination),
    )
'''

verify_script_source = r'''# Verify Notebook 04 segmentation artifacts without rerunning inference.

from __future__ import annotations

import json
from pathlib import Path

import nibabel as nib
import numpy as np

PROJECT_ROOT = Path(__file__).resolve().parents[1]
MANIFEST = PROJECT_ROOT / "data/sample_masks/notebook_04/segmentation_case_manifest.json"


def main() -> None:
    with MANIFEST.open("r", encoding="utf-8") as handle:
        manifest = json.load(handle)
    cases = manifest.get("cases", [])
    if len(cases) != 3:
        raise SystemExit(f"Expected three cases; found {len(cases)}")
    for case in cases:
        outputs = case["outputs"]
        predicted_path = PROJECT_ROOT / outputs["predicted_whole_tumor_mask_file"]
        reference_path = PROJECT_ROOT / case["reference_multiclass_mask_file"]
        predicted_image = nib.load(str(predicted_path))
        reference_image = nib.load(str(reference_path))
        predicted = np.asanyarray(predicted_image.dataobj)
        if predicted.shape != reference_image.shape:
            raise SystemExit(f"Shape mismatch for {case['case_id']}")
        if not np.allclose(predicted_image.affine, reference_image.affine, atol=1e-4):
            raise SystemExit(f"Affine mismatch for {case['case_id']}")
        if int(np.count_nonzero(predicted)) <= 0:
            raise SystemExit(f"Empty whole-tumor prediction for {case['case_id']}")
    print("Notebook 04 segmentation artifacts passed verification.")


if __name__ == "__main__":
    main()
'''

model_card = f'''# NeuroFHIR-QC Segmentation Model Card

## Model

- Bundle: `{MODEL_BUNDLE_NAME}`
- Bundle version: `{MODEL_BUNDLE_VERSION}`
- Pinned bundle commit: `{MODEL_BUNDLE_REVISION}`
- Architecture: MONAI SegResNet
- Checkpoint SHA-256: `{model_provenance['checkpoint_sha256']}`
- Input channels: T1-contrast, T1, T2, FLAIR
- Output channels: tumor core, whole tumor, enhancing tumor
- Threshold used by Notebook 04: `{MODEL_THRESHOLD}`

## Intended use

This model is used only for an academic research demonstration of the NeuroFHIR-QC
imaging-to-volumetry workflow. It is not used for diagnosis, treatment selection, or
autonomous clinical decision-making.

## Demonstration data

Notebook 04 uses three public de-identified MSD Task01 BrainTumour cases prepared in
Notebook 03. Synthetic FHIR records provide workflow context only. No real patient
identity linkage is made.

## Evaluation limitation

The public MSD/BraTS cases may share source lineage with data used to train the published
BraTS model. Therefore, the three-case results are an executable demonstration benchmark,
not independent external validation and not clinical validation.

## Postprocessing

Thresholded model channels are made hierarchically consistent using ET ⊆ TC ⊆ WT.
This is an engineering postprocessing rule and is recorded in every case result.

## Next safety stage

Notebook 05 will evaluate controlled perturbations, output stability, plausibility, and
workflow triage. Notebook 04 alone does not determine whether a result should be accepted.
'''

technical_documentation = f'''# Notebook 04 — Segmentation and Volumetry

Notebook 04 runs the pinned MONAI BraTS SegResNet model on the three public MRI cases
prepared by Notebook 03. It preserves exact model provenance, modality order, inference
configuration, runtime, predicted masks, probabilities, volumetry, evaluation metrics, and
visual overlays.

## Core outputs

- `{SEGMENTATION_MANIFEST_PATH.relative_to(PROJECT_ROOT).as_posix()}`
- `{METRICS_JSON_PATH.relative_to(PROJECT_ROOT).as_posix()}`
- `{METRICS_CSV_PATH.relative_to(PROJECT_ROOT).as_posix()}`
- `{VOLUMETRY_CSV_PATH.relative_to(PROJECT_ROOT).as_posix()}`
- `{RUNTIME_LOG_PATH.relative_to(PROJECT_ROOT).as_posix()}`
- `data/sample_masks/notebook_04/<case>/`
- `evaluation/results/notebook_04_segmentation_and_volumetry/previews/`

## Safety boundary

Outputs are research-demo artifacts. They are not final FHIR Observations, clinical
reports, or reviewed results. Notebook 05 must evaluate robustness before workflow triage,
and later notebooks must preserve preliminary status until explicit human review.
'''

SEGMENTATION_SERVICE_PATH.parent.mkdir(parents=True, exist_ok=True)
VERIFY_SCRIPT_PATH.parent.mkdir(parents=True, exist_ok=True)
REQUIREMENTS_PATH.parent.mkdir(parents=True, exist_ok=True)
MODEL_CARD_PATH.parent.mkdir(parents=True, exist_ok=True)
TECHNICAL_DOC_PATH.parent.mkdir(parents=True, exist_ok=True)

SEGMENTATION_SERVICE_PATH.write_text(
    textwrap.dedent(segmentation_service_source).strip() + "\n",
    encoding="utf-8",
)
VERIFY_SCRIPT_PATH.write_text(
    textwrap.dedent(verify_script_source).strip() + "\n",
    encoding="utf-8",
)
REQUIREMENTS_PATH.write_text(
    "\n".join(
        [
            f"monai=={runtime_versions['monai']}",
            f"torch=={runtime_versions['torch'].split('+')[0]}",
            f"nibabel=={runtime_versions['nibabel']}",
            f"numpy=={runtime_versions['numpy']}",
            f"scipy=={runtime_versions['scipy']}",
            f"pandas=={runtime_versions['pandas']}",
            f"matplotlib=={runtime_versions['matplotlib']}",
            f"huggingface_hub=={runtime_versions['huggingface_hub']}",
        ]
    ) + "\n",
    encoding="utf-8",
)
MODEL_CARD_PATH.write_text(textwrap.dedent(model_card).strip() + "\n", encoding="utf-8")
TECHNICAL_DOC_PATH.write_text(
    textwrap.dedent(technical_documentation).strip() + "\n",
    encoding="utf-8",
)

for python_path in (SEGMENTATION_SERVICE_PATH, VERIFY_SCRIPT_PATH):
    compile(python_path.read_text(encoding="utf-8"), str(python_path), "exec")

print("=" * 96)
print("✅ Reusable segmentation service and verification script created")
print("✅ Generated Python files passed syntax compilation")
print(f"📦 Segmentation requirements: {REQUIREMENTS_PATH}")
print(f"🧾 Model card: {MODEL_CARD_PATH}")
print("=" * 96)

✅ Reusable segmentation service and verification script created
✅ Generated Python files passed syntax compilation
📦 Segmentation requirements: /content/drive/MyDrive/neurofhir-qc/requirements/segmentation.txt
🧾 Model card: /content/drive/MyDrive/neurofhir-qc/model/model_card.md


In [9]:
# Cell 9 — Final audit, checksums, and notebook-manifest update

# Reload persisted evidence so the final gate does not rely only on in-memory objects.
persisted_manifest = load_json(SEGMENTATION_MANIFEST_PATH)
persisted_metrics = load_json(METRICS_JSON_PATH)

if len(persisted_manifest.get("cases", [])) != 3:
    raise AssertionError("Notebook 04 did not persist exactly three case results.")
summary = persisted_metrics.get("summary", {})
required_summary_values = {
    "case_count": 3,
    "inference_success_rate": 1.0,
    "model_output_nonempty_rate": 1.0,
    "output_shape_integrity_rate": 1.0,
    "output_affine_integrity_rate": 1.0,
    "preview_count": 3,
}
for key, expected in required_summary_values.items():
    if float(summary.get(key, -1)) != float(expected):
        raise AssertionError(
            f"Notebook 04 summary check failed for {key}: {summary.get(key)}"
        )

core_files = [
    MODEL_PROVENANCE_PATH,
    SEGMENTATION_MANIFEST_PATH,
    METRICS_JSON_PATH,
    METRICS_CSV_PATH,
    VOLUMETRY_CSV_PATH,
    RUNTIME_LOG_PATH,
    SEGMENTATION_SERVICE_PATH,
    VERIFY_SCRIPT_PATH,
    REQUIREMENTS_PATH,
    MODEL_CARD_PATH,
    TECHNICAL_DOC_PATH,
]
core_files.extend(sorted(SEGMENTATION_ROOT.glob("*/*.nii.gz")))
core_files.extend(sorted(SEGMENTATION_ROOT.glob("*/*.npz")))
core_files.extend(sorted(CASE_RESULT_ROOT.glob("*.json")))
core_files.extend(sorted(PREVIEW_ROOT.glob("*.png")))
missing_core = [
    str(path)
    for path in core_files
    if not path.exists() or path.stat().st_size == 0
]
if missing_core:
    raise FileNotFoundError(
        "Notebook 04 output evidence is missing:\n"
        + "\n".join(f" - {path}" for path in missing_core)
    )

# Revalidate every persisted whole-tumor output against its reference geometry.
geometry_rows = []
for case in persisted_manifest["cases"]:
    predicted_path = (
        PROJECT_ROOT / case["outputs"]["predicted_whole_tumor_mask_file"]
    )
    reference_path = PROJECT_ROOT / case["reference_multiclass_mask_file"]
    predicted_image = nib.load(str(predicted_path))
    reference_image = nib.load(str(reference_path))
    predicted = np.asanyarray(predicted_image.dataobj)
    shape_ok = predicted.shape == reference_image.shape
    affine_ok = np.allclose(
        predicted_image.affine, reference_image.affine, atol=1e-4, rtol=0.0
    )
    nonempty = int(np.count_nonzero(predicted)) > 0
    if not (shape_ok and affine_ok and nonempty):
        raise AssertionError(
            f"Persisted segmentation failed geometry/nonempty checks for {case['case_id']}"
        )
    geometry_rows.append(
        {
            "case_id": case["case_id"],
            "shape_match": shape_ok,
            "affine_match": affine_ok,
            "whole_tumor_nonempty": nonempty,
            "predicted_voxels": int(np.count_nonzero(predicted)),
        }
    )

# Completion is based on executed evidence. The fully executed notebook can then
# be copied to GitHub, matching the repository workflow used for Notebook 03.
notebook_saved_in_drive = (
    NOTEBOOK_SAVE_PATH.exists()
    and NOTEBOOK_SAVE_PATH.is_file()
    and NOTEBOOK_SAVE_PATH.stat().st_size > 0
)
manifest_status = "completed"

checksum_paths = sorted(
    {path.resolve() for path in core_files if path.exists() and path.is_file()},
    key=lambda path: str(path),
)
checksums = [
    {
        "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in checksum_paths
]

final_audit = {
    "project_name": project_config["project_name"],
    "project_version": project_config.get("version", "0.1.0"),
    "notebook_number": "04",
    "notebook_filename": NOTEBOOK_FILENAME,
    "status": manifest_status,
    "audited_utc": utc_now(),
    "notebook_saved_in_drive": notebook_saved_in_drive,
    "recommended_drive_notebook_path": str(NOTEBOOK_SAVE_PATH),
    "completion_basis": "Executed outputs and audit checks passed.",
    "model": {
        "bundle_name": MODEL_BUNDLE_NAME,
        "bundle_version": MODEL_BUNDLE_VERSION,
        "bundle_revision": MODEL_BUNDLE_REVISION,
        "checkpoint_sha256": model_provenance["checkpoint_sha256"],
        "input_order": ["T1c", "T1", "T2", "FLAIR"],
        "output_channels": ["tumor_core", "whole_tumor", "enhancing_tumor"],
    },
    "inference_configuration": {
        "device": str(DEVICE),
        "gpu_name": GPU_PROPERTIES.name,
        "gpu_memory_gib": round(GPU_MEMORY_GB, 3),
        "roi_size": list(ROI_SIZE),
        "sw_batch_size": SW_BATCH_SIZE,
        "overlap": SW_OVERLAP,
        "threshold": MODEL_THRESHOLD,
        "amp_enabled": USE_AMP,
    },
    "metrics": summary,
    "geometry_validation": geometry_rows,
    "scope": {
        "segmentation_inference_run": True,
        "volumetry_calculated": True,
        "reference_metrics_calculated": True,
        "visual_overlays_created": True,
        "qc_classification_calculated": False,
        "current_ai_observation_created": False,
        "human_review_transition_executed": False,
        "ai_result_writeback_performed": False,
    },
    "data_governance": {
        "public_deidentified_imaging_only": True,
        "synthetic_fhir_context_only": True,
        "real_patient_linkage_claimed": False,
        "independent_external_validation_claimed": False,
        "clinical_validation_claimed": False,
        "phi_allowed": False,
        "diagnostic_use": False,
    },
    "checksum_inventory": checksums,
    "next_notebook": (
        "05 — Trust and Robustness Engine: controlled perturbations, stability, "
        "plausibility, provenance completeness, and QC triage."
    ),
}
write_json(AUDIT_JSON_PATH, final_audit)

audit_markdown = f'''# Notebook 04 — Segmentation and Volumetry

**Notebook:** `{NOTEBOOK_FILENAME}`
**Status:** `{manifest_status}`
**Model:** `{MODEL_BUNDLE_NAME}` version `{MODEL_BUNDLE_VERSION}` at commit `{MODEL_BUNDLE_REVISION}`
**Audited UTC:** `{final_audit['audited_utc']}`

## Execution evidence

| Metric | Result |
|---|---:|
| Cases segmented | {summary['case_count']}/3 |
| Inference success | {summary['inference_success_rate']:.1%} |
| Non-empty whole-tumor outputs | {summary['model_output_nonempty_rate']:.1%} |
| Output shape integrity | {summary['output_shape_integrity_rate']:.1%} |
| Output affine integrity | {summary['output_affine_integrity_rate']:.1%} |
| Mean whole-tumor Dice | {summary['mean_whole_tumor_dice']:.4f} |
| Mean whole-tumor HD95 | {summary['mean_whole_tumor_hd95_mm']:.3f} mm |
| Mean absolute volume error | {summary['mean_absolute_whole_tumor_volume_error_ml']:.3f} mL |
| Mean relative volume error | {summary['mean_relative_whole_tumor_volume_error']:.3%} |
| Mean inference time | {summary['mean_inference_seconds']:.2f} s |
| Evaluation previews | {summary['preview_count']}/3 |

## Interpretation boundary

The three MSD cases form an executable demonstration benchmark. They are not claimed as
independent external validation or clinical validation. The images are public de-identified
research data and are linked only to synthetic FHIR demonstration context.

Notebook 04 generated model outputs and volumetry but did not assign the final QC category,
perform human review, create a final FHIR Observation, or write AI evidence to a server.

## Completion gate

All required outputs, geometry checks, metrics, previews, reusable code, and
checksums passed. The executed notebook may now be copied to GitHub with its
cell outputs preserved. Drive copy detected: **{'yes' if notebook_saved_in_drive else 'no'}**
'''
AUDIT_MD_PATH.write_text(textwrap.dedent(audit_markdown).strip() + "\n", encoding="utf-8")

notebook_04_entry["status"] = manifest_status
notebook_04_entry["last_executed_utc"] = utc_now()
notebook_04_entry["model_bundle"] = MODEL_BUNDLE_NAME
notebook_04_entry["model_bundle_revision"] = MODEL_BUNDLE_REVISION
notebook_04_entry["segmented_case_count"] = summary["case_count"]
notebook_04_entry["inference_success_rate"] = summary["inference_success_rate"]
notebook_04_entry["mean_whole_tumor_dice"] = summary["mean_whole_tumor_dice"]
notebook_04_entry["audit_path"] = AUDIT_JSON_PATH.relative_to(PROJECT_ROOT).as_posix()
write_json(NOTEBOOK_MANIFEST_PATH, notebook_manifest)

print("=" * 96)
print("✅ Notebook 04 segmentation and volumetry evidence passed")
print("✅ 3/3 public MRI cases segmented")
print("✅ Predicted masks, volumes, metrics, overlays, and provenance archived")
print(f"📊 Mean whole-tumor Dice: {summary['mean_whole_tumor_dice']:.4f}")
print(f"📏 Mean absolute volume error: {summary['mean_absolute_whole_tumor_volume_error_ml']:.3f} mL")
print(f"✅ Audit JSON: {AUDIT_JSON_PATH}")
print(f"✅ Audit Markdown: {AUDIT_MD_PATH}")
print(f"📓 Manifest status: {manifest_status}")
print("🎯 Notebook 04 is complete; copy the executed notebook to GitHub")
print("➡️ Notebook 05 may begin after the GitHub copy preserves these outputs")
print("=" * 96)

✅ Notebook 04 segmentation and volumetry evidence passed
✅ 3/3 public MRI cases segmented
✅ Predicted masks, volumes, metrics, overlays, and provenance archived
📊 Mean whole-tumor Dice: 0.9014
📏 Mean absolute volume error: 1.313 mL
✅ Audit JSON: /content/drive/MyDrive/neurofhir-qc/evaluation/results/notebook_04_segmentation_volumetry_audit.json
✅ Audit Markdown: /content/drive/MyDrive/neurofhir-qc/docs/NOTEBOOK_04_SEGMENTATION_AND_VOLUMETRY.md
📓 Manifest status: completed
🎯 Notebook 04 is complete; copy the executed notebook to GitHub
➡️ Notebook 05 may begin after the GitHub copy preserves these outputs
